# Notebook 1: MILP Single-Site BESS Optimization — Baseline

## References

1. **Krishnamurthy, D., Uckun, C., Zhou, Z., Thimmapuram, P. R., Botterud, A. (2018).** *"Energy Storage Arbitrage Under Day-Ahead and Real-Time Price Uncertainty."* IEEE Transactions on Power Systems, 33(1), 84–93. [DOI: 10.1109/TPWRS.2017.2691995](https://doi.org/10.1109/TPWRS.2017.2691995)

2. **Weitzel, T., Glock, C. H. (2018).** *"Energy management for stationary electric energy storage systems: A systematic literature review."* European Journal of Operational Research, 264(2), 582–606. [DOI: 10.1016/j.ejor.2017.06.052](https://doi.org/10.1016/j.ejor.2017.06.052)

## What these papers bring

Krishnamurthy et al. (2018) formulate a MILP for optimal BESS scheduling under day-ahead and real-time price uncertainty. The model captures charge/discharge efficiency, state-of-charge limits, and power rating constraints. Weitzel & Glock (2018) provide a comprehensive review of energy storage management techniques, categorizing MILP as the most widely used approach for deterministic single-site scheduling.

## What is implemented below

We implement a **deterministic MILP** for a single prosumer site with:
- PV generation forecast
- Consumption (load) forecast  
- Day-ahead electricity prices (mimicking OTE CZ structure with hourly resolution)
- A BESS (Battery Energy Storage System) with charge/discharge limits, round-trip efficiency, and SoC bounds

This represents the **current baseline** of how the company optimizes each customer independently. The objective is to **minimize the daily electricity cost** by shifting consumption to cheaper hours via the battery.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pulp

np.random.seed(42)


## 1. Problem Data

We generate synthetic 24-hour profiles mimicking a Czech prosumer:
- **PV generation**: bell-curve peaking around noon  
- **Load**: typical household with morning and evening peaks  
- **Day-ahead prices**: hourly OTE-style prices (EUR/MWh) with morning and evening peaks


In [ ]:
# Time horizon: 24 hours, hourly resolution
T = 24
hours = np.arange(T)

# PV generation profile (kW) — bell curve peaking at hour 12
pv = np.maximum(0, 5.0 * np.exp(-0.5 * ((hours - 12) / 3.0) ** 2))

# Load profile (kW) — two peaks (morning, evening)
load = 1.5 + 1.0 * np.exp(-0.5 * ((hours - 8) / 2.0) ** 2) + \
       1.5 * np.exp(-0.5 * ((hours - 19) / 2.5) ** 2) + \
       0.3 * np.random.randn(T)
load = np.maximum(load, 0.5)

# Day-ahead prices (EUR/MWh) — OTE-like pattern
base_price = 40 + 20 * np.sin(2 * np.pi * (hours - 6) / 24) + \
             15 * np.exp(-0.5 * ((hours - 18) / 3.0) ** 2) + \
             5 * np.random.randn(T)
price_buy = np.maximum(base_price, 10.0)   # EUR/MWh
price_sell = price_buy * 0.8                # sell-back at 80% of buy price

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(hours, pv, color='gold', label='PV')
axes[0].plot(hours, load, 'r-o', markersize=3, label='Load')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('kW')
axes[0].set_title('PV Generation & Load'); axes[0].legend()

axes[1].plot(hours, price_buy, 'b-o', markersize=3, label='Buy price')
axes[1].plot(hours, price_sell, 'g--s', markersize=3, label='Sell price')
axes[1].set_xlabel('Hour'); axes[1].set_ylabel('EUR/MWh')
axes[1].set_title('Day-Ahead Prices (OTE-style)'); axes[1].legend()

axes[2].bar(hours, pv - load, color='teal', alpha=0.7)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_xlabel('Hour'); axes[2].set_ylabel('kW')
axes[2].set_title('Net Generation (PV - Load)')

plt.tight_layout()
plt.savefig('/tmp/nb1_data.png', dpi=100)
plt.show()
print("Data generated successfully.")


## 2. MILP Formulation

### Decision Variables
- $p^{\\text{ch}}_t$: charging power at hour $t$ (kW)
- $p^{\\text{dis}}_t$: discharging power at hour $t$ (kW)
- $p^{\\text{grid,buy}}_t$: power bought from grid (kW)
- $p^{\\text{grid,sell}}_t$: power sold to grid (kW)
- $\\text{SoC}_t$: state of charge at end of hour $t$ (kWh)
- $u_t \\in \\{0, 1\\}$: binary — 1 if charging, 0 if discharging

### Objective
$$\\min \\sum_{t=1}^{T} \\left( \\pi^{\\text{buy}}_t \\cdot p^{\\text{grid,buy}}_t - \\pi^{\\text{sell}}_t \\cdot p^{\\text{grid,sell}}_t \\right) \\cdot \\Delta t$$

### Constraints
1. **Power balance**: $p^{\\text{pv}}_t + p^{\\text{dis}}_t + p^{\\text{grid,buy}}_t = p^{\\text{load}}_t + p^{\\text{ch}}_t + p^{\\text{grid,sell}}_t$
2. **SoC dynamics**: $\\text{SoC}_t = \\text{SoC}_{t-1} + \\eta^{\\text{ch}} p^{\\text{ch}}_t \\Delta t - \\frac{p^{\\text{dis}}_t \\Delta t}{\\eta^{\\text{dis}}}$
3. **SoC limits**: $\\text{SoC}_{\\min} \\leq \\text{SoC}_t \\leq \\text{SoC}_{\\max}$
4. **Charge/discharge limits**: $0 \\leq p^{\\text{ch}}_t \\leq P^{\\max} \\cdot u_t$, $0 \\leq p^{\\text{dis}}_t \\leq P^{\\max} \\cdot (1 - u_t)$
5. **Non-negativity**: all power variables $\\geq 0$


In [ ]:
# BESS Parameters
E_max = 10.0    # kWh — battery capacity
E_min = 1.0     # kWh — minimum SoC (10% DoD)
P_max = 5.0     # kW — max charge/discharge power
eta_ch = 0.95   # charging efficiency
eta_dis = 0.95  # discharging efficiency
SoC_init = 5.0  # kWh — initial SoC
dt = 1.0        # hour

# Create MILP problem
prob = pulp.LpProblem("SingleSite_BESS_Optimization", pulp.LpMinimize)

# Decision variables
p_ch = [pulp.LpVariable(f"p_ch_{t}", lowBound=0, upBound=P_max) for t in range(T)]
p_dis = [pulp.LpVariable(f"p_dis_{t}", lowBound=0, upBound=P_max) for t in range(T)]
p_buy = [pulp.LpVariable(f"p_buy_{t}", lowBound=0) for t in range(T)]
p_sell = [pulp.LpVariable(f"p_sell_{t}", lowBound=0) for t in range(T)]
soc = [pulp.LpVariable(f"soc_{t}", lowBound=E_min, upBound=E_max) for t in range(T)]
u = [pulp.LpVariable(f"u_{t}", cat='Binary') for t in range(T)]  # 1=charging

# Objective: minimize net cost
prob += pulp.lpSum([
    (price_buy[t] / 1000.0) * p_buy[t] * dt - (price_sell[t] / 1000.0) * p_sell[t] * dt
    for t in range(T)
]), "Total_Cost"

# Constraints
for t in range(T):
    # Power balance
    prob += (pv[t] + p_dis[t] + p_buy[t] == load[t] + p_ch[t] + p_sell[t],
             f"PowerBalance_{t}")
    
    # SoC dynamics
    if t == 0:
        prob += (soc[t] == SoC_init + eta_ch * p_ch[t] * dt - p_dis[t] * dt / eta_dis,
                 f"SoC_dynamics_{t}")
    else:
        prob += (soc[t] == soc[t-1] + eta_ch * p_ch[t] * dt - p_dis[t] * dt / eta_dis,
                 f"SoC_dynamics_{t}")
    
    # Charge/discharge mutual exclusion
    prob += (p_ch[t] <= P_max * u[t], f"ChargeBound_{t}")
    prob += (p_dis[t] <= P_max * (1 - u[t]), f"DischargeBound_{t}")

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Status: {pulp.LpStatus[prob.status]}")
print(f"Optimal daily cost: {pulp.value(prob.objective):.2f} EUR")


## 3. Results Visualization


In [ ]:
# Extract results
soc_vals = [pulp.value(soc[t]) for t in range(T)]
p_ch_vals = [pulp.value(p_ch[t]) for t in range(T)]
p_dis_vals = [pulp.value(p_dis[t]) for t in range(T)]
p_buy_vals = [pulp.value(p_buy[t]) for t in range(T)]
p_sell_vals = [pulp.value(p_sell[t]) for t in range(T)]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# SoC profile
axes[0, 0].fill_between(hours, E_min, soc_vals, alpha=0.3, color='blue')
axes[0, 0].plot(hours, soc_vals, 'b-o', markersize=3, label='SoC')
axes[0, 0].axhline(E_max, color='red', linestyle='--', alpha=0.5, label='SoC max')
axes[0, 0].axhline(E_min, color='red', linestyle='--', alpha=0.5, label='SoC min')
axes[0, 0].set_xlabel('Hour'); axes[0, 0].set_ylabel('kWh')
axes[0, 0].set_title('Battery State of Charge'); axes[0, 0].legend()

# Charge/Discharge
axes[0, 1].bar(hours, p_ch_vals, color='green', alpha=0.7, label='Charge')
axes[0, 1].bar(hours, [-v for v in p_dis_vals], color='red', alpha=0.7, label='Discharge')
axes[0, 1].set_xlabel('Hour'); axes[0, 1].set_ylabel('kW')
axes[0, 1].set_title('Battery Charge / Discharge'); axes[0, 1].legend()

# Grid exchange
axes[1, 0].bar(hours, p_buy_vals, color='coral', alpha=0.7, label='Grid buy')
axes[1, 0].bar(hours, [-v for v in p_sell_vals], color='teal', alpha=0.7, label='Grid sell')
axes[1, 0].set_xlabel('Hour'); axes[1, 0].set_ylabel('kW')
axes[1, 0].set_title('Grid Power Exchange'); axes[1, 0].legend()

# Cost breakdown per hour
costs = [(price_buy[t]/1000)*p_buy_vals[t]*dt - (price_sell[t]/1000)*p_sell_vals[t]*dt for t in range(T)]
axes[1, 1].bar(hours, costs, color=['coral' if c > 0 else 'teal' for c in costs], alpha=0.7)
axes[1, 1].set_xlabel('Hour'); axes[1, 1].set_ylabel('EUR')
axes[1, 1].set_title('Hourly Net Cost')

plt.tight_layout()
plt.savefig('/tmp/nb1_results.png', dpi=100)
plt.show()

# Summary
total_cost = sum(costs)
total_buy = sum(p_buy_vals)
total_sell = sum(p_sell_vals)
print(f"Total daily cost:    {total_cost:.2f} EUR")
print(f"Total grid buy:      {total_buy:.1f} kWh")
print(f"Total grid sell:     {total_sell:.1f} kWh")
print(f"Total PV generated:  {sum(pv):.1f} kWh")
print(f"Total load:          {sum(load):.1f} kWh")


## 4. Comparison: With vs Without Battery

To illustrate the value of the BESS, let's compute what the cost would be without any battery (direct grid + PV only).


In [ ]:
# Without battery: all net demand from grid, excess PV sold back
cost_no_bess = 0.0
for t in range(T):
    net = load[t] - pv[t]
    if net > 0:
        cost_no_bess += (price_buy[t] / 1000.0) * net * dt
    else:
        cost_no_bess -= (price_sell[t] / 1000.0) * (-net) * dt

print(f"Cost WITHOUT BESS: {cost_no_bess:.2f} EUR")
print(f"Cost WITH BESS:    {total_cost:.2f} EUR")
print(f"Daily savings:     {cost_no_bess - total_cost:.2f} EUR ({100*(cost_no_bess - total_cost)/abs(cost_no_bess):.1f}%)")


## 5. Key Takeaways for the Thesis

1. **This MILP formulation is the building block** for all subsequent multi-site / BRP models.
2. Each customer is optimized **independently** — there is no coordination between sites.
3. The battery shifts consumption from expensive hours to cheap hours and captures PV surplus.
4. Moving to a BRP model requires **aggregating** multiple such sites and jointly optimizing their grid exchange to minimize portfolio imbalance costs.
5. The **sell price < buy price** spread creates a natural incentive for self-consumption, which will interact with the BRP's nomination strategy.

---

*This notebook serves as the baseline against which all BRP / aggregation approaches will be compared.*
